In [ ]:
import os
import pandas as pd


In [5]:

# ------------------------------------------------------------------
# CONFIGURACIÓN: editá estas tres variables según lo que necesites
# ------------------------------------------------------------------
ARCHIVO = "ENERGY.xls"          # nombre (o ruta) del archivo Excel
COLUMNA = "Kilowatt Hours"              # nombre de la columna a analizar
CANTIDAD_INTERVALOS = 5         # cantidad de intervalos de la tabla
# ------------------------------------------------------------------


In [7]:
def generar_tabla_frecuencias(archivo: str, columna: str, cantidad_intervalos: int) -> pd.DataFrame:
    ruta = archivo if os.path.isabs(archivo) else os.path.join(os.getcwd(), archivo)

    if not os.path.exists(ruta):
        raise FileNotFoundError(f"No se encontró el archivo: {ruta}")

    df = pd.read_excel(ruta)

    if columna not in df.columns:
        raise ValueError(
            f"La columna '{columna}' no existe en el archivo. "
            f"Columnas disponibles: {list(df.columns)}"
        )

    datos = pd.to_numeric(df[columna], errors="coerce").dropna()

    if datos.empty:
        raise ValueError(f"La columna '{columna}' no tiene datos numéricos.")

    # Intervalos de igual amplitud
    conteo = pd.cut(datos, bins=cantidad_intervalos, include_lowest=True).value_counts(sort=False)
    conteo = conteo.sort_index()

    tabla = pd.DataFrame({
        "Clases": [str(intervalo) for intervalo in conteo.index],
        "Marca de clase": [round(intervalo.mid, 2) for intervalo in conteo.index],
        "Frecuencia absoluta": conteo.values,
    })

    tabla["Frecuencia relativa"] = (tabla["Frecuencia absoluta"] / tabla["Frecuencia absoluta"].sum()).round(4)
    tabla["Frecuencia acumulada"] = tabla["Frecuencia absoluta"].cumsum()
    tabla["Frecuencia porcentual"] = (tabla["Frecuencia relativa"] * 100).round(2)
    tabla["Frecuencia acumulada porcentual"] = tabla["Frecuencia porcentual"].cumsum().round(2)

    # Fila de totales (no aplica a las frecuencias acumuladas)
    fila_total = {
        "Clases": "Total",
        "Marca de clase": "",
        "Frecuencia absoluta": tabla["Frecuencia absoluta"].sum(),
        "Frecuencia relativa": tabla["Frecuencia relativa"].sum(),
        "Frecuencia acumulada": "",
        "Frecuencia porcentual": tabla["Frecuencia porcentual"].sum(),
        "Frecuencia acumulada porcentual": "",
    }
    tabla = pd.concat([tabla, pd.DataFrame([fila_total])], ignore_index=True)

    return tabla


def main():
    tabla = generar_tabla_frecuencias(ARCHIVO, COLUMNA, CANTIDAD_INTERVALOS)

    print("\nTabla de frecuencias:\n")
    print(tabla.to_string(index=False))


if __name__ == "__main__":
    main()


Tabla de frecuencias:

                       Clases Marca de clase  Frecuencia absoluta  Frecuencia relativa Frecuencia acumulada  Frecuencia porcentual Frecuencia acumulada porcentual
(6377.8279999999995, 10030.2]        8204.01                   13               0.2549                   13                  25.49                           25.49
           (10030.2, 13664.4]        11847.3                   14               0.2745                   27                  27.45                           52.94
           (13664.4, 17298.6]        15481.5                   17               0.3333                   44                  33.33                           86.27
           (17298.6, 20932.8]        19115.7                    6               0.1176                   50                  11.76                           98.03
           (20932.8, 24567.0]        22749.9                    1               0.0196                   51                   1.96                           99.9